In [1]:
!pip install transformers
!pip install datasets

In [2]:
import random
import pandas as pd
import numpy as np

# =============================
# Reproducibility
# =============================
random.seed(42)
np.random.seed(42)

# =============================
# Main dataset text (UNIQUE)
# =============================
A_texts = [
    "App crashes when I open settings",
    "Battery drains faster after update",
    "Login fails intermittently",
    "Smooth experience overall",
    "Payment page freezes suddenly",
    "Unexpected error occurs sometimes",
    "Works fine on my device",
    "Performance is slow after installation",
    "App behaves inconsistently across devices",
    "User interface becomes unresponsive randomly"
]

# =============================
# Categorical values
# =============================
b_vals = ["UI", "Performance", "Auth", "Payment", "UX"]
c_vals = ["Android", "iOS", "Web"]
d_vals = ["Crash", "Battery", "Error", "Freeze", "General"]

# =============================
# LONG e descriptions (lookup only)
# =============================
e_descriptions = [
    "Issue observed on Android devices with low memory when multiple background apps are running",
    "Occurs intermittently after recent update when network connectivity fluctuates",
    "Reported mainly by users using older devices during extended usage sessions",
    "Reproduced when accessibility services are enabled along with dark mode",
    "Happens only when user navigates rapidly between multiple screens",
    "Observed after app remains idle in background for a long duration",
    "Occurs under heavy load when system resources are constrained",
    "Reported during peak usage hours and disappears after restart",
    "Seen when biometric authentication is enabled together with auto-login",
    "Triggered when switching between WiFi and mobile data during critical operations",
    "Occurs only on specific device models running older operating system versions",
    "Observed when multiple user accounts are logged in simultaneously"
]

# =============================
# Helper
# =============================
def bin_to_str(label):
    return "positive" if label == 1 else "negative"

# =============================
# Generate Lookup Table (unchanged, noisy, larger)
# =============================
lut_rows = []

all_combinations = [(b, c, d) for b in b_vals for c in c_vals for d in d_vals]
base_combinations = random.sample(all_combinations, k=20)

for b, c, d in base_combinations:
    num_e = random.randint(3, 5)
    chosen_es = random.sample(e_descriptions, num_e)

    base_prob = 0.7 if d in ["Crash", "Error", "Freeze"] else 0.3

    for i, e in enumerate(chosen_es):
        t_bin = np.random.binomial(1, base_prob) if i % 2 == 0 else 1 - np.random.binomial(1, base_prob)
        lut_rows.append([b, c, d, e, bin_to_str(t_bin)])

lookup_df = pd.DataFrame(lut_rows, columns=["b", "c", "d", "e", "t"])

# =============================
# Generate Main Dataset (SMALL, UNIQUE A)
# =============================
main_rows = []

for A in A_texts:
    b = random.choice(b_vals)
    c = random.choice(c_vals)
    d = random.choice(d_vals)

    lut_match = lookup_df[
        (lookup_df.b == b) &
        (lookup_df.c == c) &
        (lookup_df.d == d)
    ]

    if len(lut_match) > 0:
        p = (lut_match.t == "positive").mean()
        t_bin = np.random.binomial(1, p)
    else:
        t_bin = np.random.binomial(1, 0.4)

    main_rows.append([A, b, c, d, bin_to_str(t_bin)])

main_df = pd.DataFrame(main_rows, columns=["A", "b", "c", "d", "t"])

# =============================
# Save
# =============================
main_df.to_csv("main_dataset.csv", index=False)
lookup_df.to_csv("lookup_table.csv", index=False)

# =============================
# Sanity checks
# =============================
print("Main dataset shape:", main_df.shape)   # should be (10, 5)
print("Lookup table shape:", lookup_df.shape)
print("A unique:", main_df["A"].is_unique)

Main dataset shape: (10, 5)
Lookup table shape: (83, 5)
A unique: True


In [3]:
df_lookup = pd.read_csv("/content/lookup_table.csv")
df_main = pd.read_csv("/content/main_dataset.csv")

In [4]:
df_lookup.head(10)

,b,c,d,e,t
0,UI,Web,General,Observed when multiple user accounts are logge...,negative
1,UI,Web,General,Seen when biometric authentication is enabled ...,negative
2,UI,Web,General,Occurs under heavy load when system resources ...,positive
3,UI,Web,General,Reproduced when accessibility services are ena...,positive
4,UI,Web,General,Reported during peak usage hours and disappear...,negative
5,UI,Android,Freeze,Happens only when user navigates rapidly betwe...,positive
6,UI,Android,Freeze,Issue observed on Android devices with low mem...,negative
7,UI,Android,Freeze,Reported mainly by users using older devices d...,negative
8,UI,Android,Freeze,Occurs under heavy load when system resources ...,negative
9,UI,Android,Freeze,Observed after app remains idle in background ...,negative


In [5]:
joined_df = main_df.merge(
    lookup_df,
    on=["b", "c", "d"],
    how="left",
    suffixes=("_main", "_lookup")
)



In [6]:
joined_df

,A,b,c,d,t_main,e,t_lookup
0,App crashes when I open settings,Performance,Web,Freeze,negative,Triggered when switching between WiFi and mobi...,positive
1,App crashes when I open settings,Performance,Web,Freeze,negative,Happens only when user navigates rapidly betwe...,negative
2,App crashes when I open settings,Performance,Web,Freeze,negative,Issue observed on Android devices with low mem...,positive
3,App crashes when I open settings,Performance,Web,Freeze,negative,Reported during peak usage hours and disappear...,negative
4,Battery drains faster after update,UX,Android,Freeze,negative,NaN,NaN
5,Login fails intermittently,Payment,Web,Freeze,negative,NaN,NaN
6,Smooth experience overall,UX,iOS,General,positive,NaN,NaN
7,Payment page freezes suddenly,UI,Web,Crash,positive,NaN,NaN
8,Unexpected error occurs sometimes,UX,iOS,Error,negative,Occurs only on specific device models running ...,positive
9,Unexpected error occurs sometimes,UX,iOS,Error,negative,Reported mainly by users using older devices d...,negative


In [7]:
len(joined_df)

17

In [8]:
joined_df["e"] = joined_df["e"].fillna("")
joined_df["t_lookup"] = joined_df["t_lookup"].fillna("")

In [9]:
train_df = (
    joined_df
    .groupby("A", as_index=False)
    .agg({
        "b": "first",
        "c": "first",
        "d": "first",
        "t_main": "first",
        "e": lambda x: " || ".join([i for i in x if i != ""]),
        "t_lookup": lambda x: " || ".join([i for i in x if i != ""])
    })
)

In [10]:
len(train_df)

10

In [11]:
df = train_df

In [12]:
train_df

,A,b,c,d,t_main,e,t_lookup
0,App behaves inconsistently across devices,Auth,Web,Battery,positive,,
1,App crashes when I open settings,Performance,Web,Freeze,negative,Triggered when switching between WiFi and mobi...,positive || negative || positive || negative
2,Battery drains faster after update,UX,Android,Freeze,negative,,
3,Login fails intermittently,Payment,Web,Freeze,negative,,
4,Payment page freezes suddenly,UI,Web,Crash,positive,,
5,Performance is slow after installation,Performance,iOS,Crash,negative,,
6,Smooth experience overall,UX,iOS,General,positive,,
7,Unexpected error occurs sometimes,UX,iOS,Error,negative,Occurs only on specific device models running ...,positive || negative || positive || negative |...
8,User interface becomes unresponsive randomly,UX,Android,Error,positive,,
9,Works fine on my device,UI,iOS,Freeze,negative,,


In [13]:
def split_safe(x):
    if pd.isna(x) or x == "":
        return []
    return [i.strip() for i in str(x).split("||") if i.strip()]

In [14]:
df["e_list"] = df["e"].apply(split_safe)
df["t_lookup_list"] = df["t_lookup"].apply(split_safe)

In [15]:
def interleave_evidence_history(e_list, h_list):
    max_len = max(len(e_list), len(h_list))
    pairs = []

    for i in range(max_len):
        e = e_list[i] if i < len(e_list) else ""
        h = h_list[i] if i < len(h_list) else ""

        if e or h:
            pairs.append(f"#Rule {i+1}: {e} #Category {i+1}: {h}")

    return " ".join(pairs)

In [16]:
df["EH_context"] = df.apply(
    lambda r: interleave_evidence_history(r["e_list"], r["t_lookup_list"]),
    axis=1
)

In [17]:
df["text"] = df["A"]

df["Context"] = (
    "[CAT] " + df["b"] +
    " [PLAT] " + df["c"] +
    " [TYPE] " + df["d"] +
    " [PAIR] " + df["EH_context"]
)

df["label"] = df["t_main"]

In [18]:
from datasets import Dataset
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction,
    set_seed
)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report
)
import argparse
import os
import json

In [19]:
# parser = argparse.ArgumentParser()
# parser.add_argument('--f1', type=int, default=1)
# parser.add_argument('--f2', type=int, default=1)
# parser.add_argument('--fi', type=int, default=0)
# parser.add_argument('--s', type=int, default=42)
# parser.add_argument('--lr', type=float, default=2e-5)
# parser.add_argument('--b', type=int, default=16)
# parser.add_argument('--e', type=int, default=10)

# args, _ = parser.parse_known_args()

seed = 42
learning_rate = 2e-5
batch_size = 4
epochs = 3
fold_index = 0
flag = 1
flag2 = 1

set_seed(seed)

In [20]:
model_name = "bert-base-uncased"
folder = f"./bert-bin-b{batch_size}/"

In [21]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_length = 512

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [22]:
LABEL2ID = {"negative": 0, "positive": 1}
ID2LABEL = {0: "negative", 1: "positive"}

In [23]:
def preprocess_data(examples):
    encoding = tokenizer(
        examples["text"],
        examples["Context"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    encoding["labels"] = [LABEL2ID[l] for l in examples["label"]]
    return encoding

In [24]:
def compute_metrics(p: EvalPrediction):
    logits = p.predictions
    labels = p.label_ids

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(labels, probs[:, 1])
    }

In [25]:
def train(train_df, test_df, fold):

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    train_ds = Dataset.from_pandas(train_df)
    test_ds = Dataset.from_pandas(test_df)

    train_ds = train_ds.map(preprocess_data, batched=True, remove_columns=train_ds.column_names)
    test_ds = test_ds.map(preprocess_data, batched=True, remove_columns=test_ds.column_names)

    train_ds.set_format("torch")
    test_ds.set_format("torch")

    args = TrainingArguments(
        output_dir=f"{folder}/{fold}",
        eval_strategy="epoch",  # Updated from evaluation_strategy
        save_strategy="epoch",  # Updated from save_strategy
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none"
    )





    # evaluation_strategy="epoch",
    # save_strategy="epoch",

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        processing_class = tokenizer,
        compute_metrics=compute_metrics
    )



    # tokenizer=tokenizer,

    trainer.train()
    eval_metrics = trainer.evaluate()

    with open(f"{folder}/{fold}/results.json", "w") as f:
        json.dump(eval_metrics, f, indent=4)

    model.save_pretrained(f"{folder}/{fold}/model")
    tokenizer.save_pretrained(f"{folder}/{fold}/model")

    del model
    torch.cuda.empty_cache()

In [26]:


kf = KFold(n_splits=5, shuffle=True, random_state=seed)

for i, (train_idx, test_idx) in enumerate(kf.split(df)):
    if flag2 == 0 and i != fold_index:
        continue

    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]

    print(f"\nFold {i}")
    print("Train:", len(train_df), "Test:", len(test_df))

    train(train_df, test_df, i)


Fold 0
Train: 8 Test: 2


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]